# TP cœur réseau — ROV : état de validation et politique

Objectif : séparer autorisation d'origine, classification et décision de
politique. VRP synthétique : AS64500, préfixe 203.0.113.0/24, longueur maximale 24.
Examiner origine correcte, origine incorrecte, sous-préfixe trop long et
préfixe non couvert, avant et après activation d'une politique explicite.

Ce modèle utilise des VRP déjà fournis. Il ne vérifie aucun certificat RPKI,
ne contacte aucun cache, n'établit aucune session BGP et n'envoie aucune annonce.
« Unavailable » est un état de disponibilité local, **pas** un quatrième état
normatif de validation. Il est séparé de NotFound pour éviter leur confusion.

Prérequis : Python 3.10+, pandas/Jupyter dans le projet verrouillé
`machine_learning_labs`. Aucune permission réseau privilégiée n'est nécessaire.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

book_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "core_network_labs" / "inputs" / "scenarios.json").is_file()
    ),
    None,
)
if book_root is None:
    raise RuntimeError(
        "Ouvrir ce notebook depuis le dossier du livre ou un de ses sous-dossiers."
    )
if str(book_root) not in sys.path:
    sys.path.insert(0, str(book_root))
from core_network_labs.run_labs import build_results  # noqa: E402 - après résolution du dossier du livre

In [ ]:
results = build_results()["bgp_rov.json"]
summary = pd.DataFrame(results["routes"])
summary

In [ ]:
assert list(summary.validation_state) == ["Valid", "Invalid", "Invalid", "NotFound"]
assert (summary.classification_only == "accept").all()
assert list(summary.reject_invalid_policy) == ["accept", "reject", "reject", "accept"]
assert results["cache_unavailable"]["validation_state"] == "Unavailable"
pd.DataFrame([results["cache_unavailable"]])

## Interprétation, politiques et limites

Classer une route n'impose pas de la rejeter : la différence observable vient
de la politique explicitement activée. NotFound n'est pas Invalid. La politique
démontrée accepte NotFound et suspend la décision quand le cache est indisponible ;
c'est un **choix local du TP**, pas une recommandation universelle.

Ajouter un second VRP qui autorise le /25 et expliquer pourquoi une autorisation
valide parmi les VRP couvrants suffit. Essayer IPv6. Distinguer ensuite une
indisponibilité du cache, un cache vide mais valide et un AS_PATH falsifié :
ce TP ne valide pas l'ensemble du chemin AS.

Source principale : RFC 8481 §5 ; classification : RFC 6811. Entrées :
`core_network_labs/inputs/scenarios.json` ; résultat attendu :
`core_network_labs/results/bgp_rov.json`. Un redémarrage du noyau remet le
scénario à zéro ; aucun état de routeur n'est créé.
